# Multi-Person Pose Estimation Demo
## AIST Group Dance Videos (sGR — 3 dancers)

**Task:** Qualitative evaluation of 3 pose estimation models on multi-person dance videos  
**Videos:** 10 dance genres × group sequences (3 dancers per video)  
**Models:** YOLO26s-pose, RTMPose-m, ViTPose-base  

**Note:** No ground truth keypoint annotations exist for AIST group sequences.  
Evaluation is qualitative (visual) + quantitative (FPS, confidence, detection count).

**Metrics:**
- FPS — inference speed
- Avg detections per frame — how many dancers detected
- Avg confidence — detection confidence
- Visual quality — skeleton correctness

**Architecture comparison:**

| Model | Type | Multi-person mechanism |
|---|---|---|
| YOLO26s | Single-stage | Detects all people in one pass |
| RTMPose-m | Top-down | Detector → crop per person → keypoints |
| ViTPose-base | Top-down | Detector → crop per person → keypoints |

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
!pip install ultralytics --quiet
!pip install rtmlib --quiet
!pip install transformers --quiet
!pip install opencv-python-headless matplotlib pandas --quiet
 
import torch
!pip uninstall onnxruntime onnxruntime-gpu -y --quiet
!pip install onnxruntime-gpu==1.19.0 --quiet
 
import onnxruntime as ort
providers = ort.get_available_providers()
backend_device = 'cuda' if 'CUDAExecutionProvider' in providers else 'cpu'
print(f'ORT backend: {backend_device}')
print('All packages installed')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import os
import cv2
import numpy as np
import time
import json
 
GROUP_DIR   = '/content/drive/MyDrive/AIST_GROUP_MP4'
RESULTS_DIR = '/content/drive/MyDrive/AIST_MultiPerson_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs('/content/frames_group', exist_ok=True)
 
videos = sorted([f for f in os.listdir(GROUP_DIR) if f.endswith('.mp4')])
print(f'Group videos: {len(videos)}')
for v in videos:
    print(f'  {v}')


In [ ]:
FRAME_STEP  = 6
MAX_FRAMES  = 50
 
video_frames = {}
 
for video_file in videos:
    video_path = f'{GROUP_DIR}/{video_file}'
    video_name = video_file.replace('.mp4', '')
    genre = video_file[:3]
 
    cap = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
 
    frame_paths = []
    frame_idx   = 0
    saved       = 0
 
    while cap.isOpened() and saved < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % FRAME_STEP == 0:
            path = f'/content/frames_group/{video_name}_f{saved:04d}.jpg'
            cv2.imwrite(path, frame)
            frame_paths.append(path)
            saved += 1
        frame_idx += 1
 
    cap.release()
    video_frames[video_name] = frame_paths
    print(f'{video_name[:20]:25s}: {saved} frames | {w}x{h} | {fps:.0f}fps')
 
total_frames = sum(len(v) for v in video_frames.values())
print(f'\nTotal frames: {total_frames}')


In [ ]:
from ultralytics import YOLO
from rtmlib import Body
 
yolo_model = YOLO('yolo26s-pose.pt')
print('YOLO26s loaded')
 
rtm_model = Body(
    det='https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_m_8xb8-300e_humanart-c2c7a14a.zip',
    det_input_size=(640, 640),
    pose='https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip',
    pose_input_size=(192, 256),
    to_openpose=False,
    backend='onnxruntime',
    device=backend_device
)
print(f'RTMPose-m loaded on {backend_device}')
 
from transformers import AutoModelForPoseEstimation, AutoProcessor
from PIL import Image
 
vit_processor = AutoProcessor.from_pretrained('usyd-community/vitpose-base-simple')
vit_model     = AutoModelForPoseEstimation.from_pretrained('usyd-community/vitpose-base-simple')
vit_model     = vit_model.to(device)
vit_model.eval()
print(f'ViTPose loaded on {device}')


In [ ]:
yolo_results = {}
yolo_times   = []
 
print('YOLO26s inference on group videos...')
 
for video_name, frame_paths in video_frames.items():
    genre = video_name[:3]
    video_stats = []
 
    for path in frame_paths:
        start = time.time()
        res   = yolo_model(path, verbose=False)
        t     = time.time() - start
        yolo_times.append(t)
 
        n_det  = len(res[0].boxes) if res[0].boxes is not None else 0
        conf   = float(res[0].boxes.conf.mean().cpu()) if n_det > 0 else 0.0
 
        video_stats.append({
            'n_det': n_det,
            'conf':  conf,
            'time':  t
        })
 
    yolo_results[video_name] = video_stats
    avg_det  = np.mean([s['n_det'] for s in video_stats])
    avg_conf = np.mean([s['conf']  for s in video_stats])
    print(f'  {genre}: avg_detections={avg_det:.1f} | avg_conf={avg_conf:.3f}')
 
print(f'\nYOLO FPS: {1/np.mean(yolo_times):.1f}')


In [ ]:
rtm_results = {}
rtm_times   = []
 
print('RTMPose inference on group videos...')
 
for video_name, frame_paths in video_frames.items():
    genre = video_name[:3]
    video_stats = []
 
    for path in frame_paths:
        img   = cv2.imread(path)
        start = time.time()
        kpts, scores = rtm_model(img)
        t = time.time() - start
        rtm_times.append(t)
 
        n_det  = len(kpts) if kpts is not None else 0
        conf   = float(np.mean(scores)) if n_det > 0 else 0.0
 
        video_stats.append({
            'n_det': n_det,
            'conf':  conf,
            'time':  t
        })
 
    rtm_results[video_name] = video_stats
    avg_det  = np.mean([s['n_det'] for s in video_stats])
    avg_conf = np.mean([s['conf']  for s in video_stats])
    print(f'  {genre}: avg_detections={avg_det:.1f} | avg_conf={avg_conf:.3f}')
 
print(f'\nRTMPose FPS: {1/np.mean(rtm_times):.1f}')


In [ ]:
vit_results = {}
vit_times   = []
 
print('ViTPose inference on group videos (YOLO bbox)...')
 
for video_name, frame_paths in video_frames.items():
    genre = video_name[:3]
    video_stats = []
 
    for frame_idx, path in enumerate(frame_paths):
        
        yolo_res = yolo_model(path, verbose=False)
        if yolo_res[0].boxes is None or len(yolo_res[0].boxes) == 0:
            video_stats.append({'n_det': 0, 'conf': 0.0, 'time': 0.0})
            continue
 
        
        boxes_xyxy = yolo_res[0].boxes.xyxy.cpu().numpy()
        boxes_xywh = []
        for box in boxes_xyxy:
            x1, y1, x2, y2 = box
            boxes_xywh.append([float(x1), float(y1), float(x2-x1), float(y2-y1)])
 
        
        img_pil = Image.open(path).convert('RGB')
        all_confs = []
 
        start = time.time()
        for bbox in boxes_xywh:
            boxes_input = [[[float(b) for b in bbox]]]
            inputs = vit_processor(img_pil, boxes=boxes_input, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = vit_model(**inputs)
            res = vit_processor.post_process_pose_estimation(outputs, boxes_input)[0][0]
            all_confs.append(float(res['scores'].mean().cpu()))
        t = time.time() - start
        vit_times.append(t)
 
        video_stats.append({
            'n_det': len(boxes_xywh),
            'conf':  float(np.mean(all_confs)),
            'time':  t
        })
 
    vit_results[video_name] = video_stats
    avg_det  = np.mean([s['n_det'] for s in video_stats])
    avg_conf = np.mean([s['conf']  for s in video_stats])
    print(f'  {genre}: avg_detections={avg_det:.1f} | avg_conf={avg_conf:.3f}')
 
print(f'\nViTPose FPS (full pipeline): {1/np.mean(vit_times):.1f}')


In [ ]:
import pandas as pd
 
GENRE_NAMES = {
    'gBR': 'Break', 'gPO': 'Pop', 'gLO': 'Lock',
    'gWA': 'Waack', 'gMH': 'Mid Hip-hop', 'gLH': 'LA Hip-hop',
    'gHO': 'House', 'gKR': 'Krump', 'gJS': 'Street Jazz', 'gJB': 'Ballet Jazz'
}
 
rows = []
for video_name in video_frames.keys():
    genre_code = video_name[:3]
    genre_name = GENRE_NAMES.get(genre_code, genre_code)
 
    yolo_stats = yolo_results[video_name]
    rtm_stats  = rtm_results[video_name]
    vit_stats  = vit_results.get(video_name, [])
 
    rows.append({
        'Genre':         genre_name,
        'YOLO Det':      round(np.mean([s['n_det'] for s in yolo_stats]), 1),
        'YOLO Conf':     round(np.mean([s['conf']  for s in yolo_stats]), 3),
        'RTM Det':       round(np.mean([s['n_det'] for s in rtm_stats]),  1),
        'RTM Conf':      round(np.mean([s['conf']  for s in rtm_stats]),  3),
        'ViT Det':       round(np.mean([s['n_det'] for s in vit_stats]),  1) if vit_stats else 0,
        'ViT Conf':      round(np.mean([s['conf']  for s in vit_stats]),  3) if vit_stats else 0,
    })
 
df = pd.DataFrame(rows)
 
print('=== MULTI-PERSON RESULTS (3 dancers per video) ===')
print(f'Expected detections per frame: 3')
print()
print(df.to_string(index=False))
 
print(f'\n=== FPS ===')
print(f'YOLO26s:   {1/np.mean(yolo_times):.1f} FPS')
print(f'RTMPose-m: {1/np.mean(rtm_times):.1f} FPS')
print(f'ViTPose:   {1/np.mean(vit_times):.1f} FPS (YOLO det + ViT pose)')
 
df.to_csv('multiperson_results.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
 
SKELETON_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(6,8),(7,9),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(12,14),(13,15),(14,16)
]
 
COLORS = ['#FF0000', '#00FF00', '#0000FF', '#FF00FF', '#FFFF00']
 
def draw_skeletons_yolo(ax, img_path, model, title):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
 
    res = model(img_path, verbose=False)
    if res[0].keypoints is None:
        ax.set_title(f'{title}\n0 detected')
        ax.axis('off')
        return
 
    n = len(res[0].keypoints)
    for p_idx in range(n):
        color = COLORS[p_idx % len(COLORS)]
        kpts  = res[0].keypoints[p_idx].xy[0].cpu().numpy()
        confs = res[0].keypoints[p_idx].conf[0].cpu().numpy()
 
        for i, j in SKELETON_PAIRS:
            if confs[i] > 0.3 and confs[j] > 0.3:
                ax.plot([kpts[i][0], kpts[j][0]],
                        [kpts[i][1], kpts[j][1]],
                        color=color, linewidth=2)
        for k in range(17):
            if confs[k] > 0.3:
                ax.scatter(kpts[k][0], kpts[k][1],
                          c=color, s=20, zorder=5)
 
    ax.set_title(f'{title}\n{n} detected', fontsize=9)
    ax.axis('off')
 
 
def draw_skeletons_rtm(ax, img_path, model, title):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
 
    kpts_all, scores_all = model(img)
    n = len(kpts_all) if kpts_all is not None else 0
 
    for p_idx in range(n):
        color = COLORS[p_idx % len(COLORS)]
        kpts  = kpts_all[p_idx]
        confs = scores_all[p_idx]
 
        for i, j in SKELETON_PAIRS:
            if confs[i] > 0.3 and confs[j] > 0.3:
                ax.plot([kpts[i][0], kpts[j][0]],
                        [kpts[i][1], kpts[j][1]],
                        color=color, linewidth=2)
        for k in range(17):
            if confs[k] > 0.3:
                ax.scatter(kpts[k][0], kpts[k][1],
                          c=color, s=20, zorder=5)
 
    ax.set_title(f'{title}\n{n} detected', fontsize=9)
    ax.axis('off')
 
 
sample_videos = list(video_frames.keys())[:3]
 
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
 
for row, video_name in enumerate(sample_videos):
    genre_code = video_name[:3]
    genre_name = GENRE_NAMES.get(genre_code, genre_code)
 
    
    frame_n = min(10, len(video_frames[video_name])-1)
    frame_path = video_frames[video_name][frame_n]
 
    draw_skeletons_yolo(axes[row, 0], frame_path, yolo_model,
                        f'YOLO26s\n{genre_name}')
    draw_skeletons_rtm(axes[row, 1], frame_path, rtm_model,
                       f'RTMPose-m\n{genre_name}')
 
    
    img = cv2.cvtColor(cv2.imread(frame_path), cv2.COLOR_BGR2RGB)
    axes[row, 2].imshow(img)
 
    yolo_res = yolo_model(frame_path, verbose=False)
    n_vit = 0
    if yolo_res[0].boxes is not None and len(yolo_res[0].boxes) > 0:
        boxes_xyxy = yolo_res[0].boxes.xyxy.cpu().numpy()
        img_pil = Image.open(frame_path).convert('RGB')
 
        for p_idx, box in enumerate(boxes_xyxy):
            x1, y1, x2, y2 = box
            bbox = [float(x1), float(y1), float(x2-x1), float(y2-y1)]
            boxes_input = [[[float(b) for b in bbox]]]
 
            inputs = vit_processor(img_pil, boxes=boxes_input, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = vit_model(**inputs)
            res = vit_processor.post_process_pose_estimation(outputs, boxes_input)[0][0]
 
            kpts  = res['keypoints'].cpu().numpy()
            confs = res['scores'].cpu().numpy()
            color = COLORS[p_idx % len(COLORS)]
 
            for i, j in SKELETON_PAIRS:
                if confs[i] > 0.3 and confs[j] > 0.3:
                    axes[row, 2].plot([kpts[i][0], kpts[j][0]],
                                      [kpts[i][1], kpts[j][1]],
                                      color=color, linewidth=2)
            for k in range(17):
                if confs[k] > 0.3:
                    axes[row, 2].scatter(kpts[k][0], kpts[k][1],
                                        c=color, s=20, zorder=5)
            n_vit += 1
 
    axes[row, 2].set_title(f'ViTPose-base\n{genre_name}\n{n_vit} detected', fontsize=9)
    axes[row, 2].axis('off')
 
plt.suptitle('Multi-Person Pose Estimation — AIST Group Dance (3 dancers)',
             fontsize=14)
plt.tight_layout()
plt.savefig('multiperson_visual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visual saved')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']
models = ['YOLO26s', 'RTMPose-m', 'ViTPose-b']
 
genres = df['Genre'].tolist()
x = range(len(genres))
w = 0.25
axes[0].bar([i-w for i in x], df['YOLO Det'], w, label='YOLO26s',   color=colors[0], alpha=0.9)
axes[0].bar([i   for i in x], df['RTM Det'],  w, label='RTMPose-m', color=colors[1], alpha=0.9)
axes[0].bar([i+w for i in x], df['ViT Det'],  w, label='ViTPose-b', color=colors[2], alpha=0.9)
axes[0].axhline(y=3, color='red', linestyle='--', label='Expected (3)')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(genres, rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('Avg detections per frame')
axes[0].set_title('Detections per Frame\n(Expected: 3 dancers)')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)
 
axes[1].bar([i-w for i in x], df['YOLO Conf'], w, label='YOLO26s',   color=colors[0], alpha=0.9)
axes[1].bar([i   for i in x], df['RTM Conf'],  w, label='RTMPose-m', color=colors[1], alpha=0.9)
axes[1].bar([i+w for i in x], df['ViT Conf'],  w, label='ViTPose-b', color=colors[2], alpha=0.9)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(genres, rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('Avg confidence')
axes[1].set_title('Detection Confidence per Genre')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)
 
fps_vals = [
    1/np.mean(yolo_times),
    1/np.mean(rtm_times),
    1/np.mean(vit_times)
]
bars = axes[2].bar(models, fps_vals, color=colors, alpha=0.9)
for bar, v in zip(bars, fps_vals):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+0.2,
                f'{v:.1f}', ha='center', fontweight='bold')
axes[2].set_ylabel('FPS')
axes[2].set_title('Inference Speed (GPU T4)')
axes[2].grid(axis='y', alpha=0.3)
 
plt.suptitle('Multi-Person Pose Estimation — AIST Group Dance Analysis',
             fontsize=13)
plt.tight_layout()
plt.savefig('multiperson_charts.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import shutil
 
files = [
    'multiperson_results.csv',
    'multiperson_visual.png',
    'multiperson_charts.png'
]
 
for f in files:
    shutil.copy(f, f'{RESULTS_DIR}/{f}')
    print(f'Saved: {f}')
 
print(f'\nAll results → AIST_MultiPerson_Results/')
print()
print('=== SUMMARY ===')
print(f'Videos: {len(videos)} group dance sequences')
print(f'Dancers per video: 3')
print(f'Expected detections: 3 per frame')
print()
print(df[['Genre','YOLO Det','RTM Det','ViT Det']].to_string(index=False))
print()
print(f'YOLO FPS:    {1/np.mean(yolo_times):.1f}')
print(f'RTMPose FPS: {1/np.mean(rtm_times):.1f}')
print(f'ViTPose FPS: {1/np.mean(vit_times):.1f}')
